# Día 3: clasificación de `dedo` con modelos fundacionales

## Objetivo

En este cuaderno vais a explorar dos usos distintos de modelos fundacionales para resolver la tarea `dedo`:

- clasificación multimodal zero-shot mediante prompting con OpenRouter;
- extracción de embeddings visuales con un encoder preentrenado y entrenamiento de un clasificador sencillo encima.

La meta no es encontrar una única solución correcta, sino comparar qué se puede conseguir:

- sin entrenar un modelo sobre vuestras clases;
- reutilizando representaciones visuales ya aprendidas.

## Interpretación del problema

En este dataset:

- `si` significa postura correcta;
- `no` significa postura incorrecta.

## Instalación

```bash
pip install openai python-dotenv pandas scikit-learn seaborn matplotlib tqdm pillow timm torch torchvision tenacity
```

## Claves API

Las claves se leen de un archivo `.env`. Añade al menos:

```text
OPENROUTER_API_KEY=sk-or-...
```

## Importaciones

In [29]:
import os
import base64
import io
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from dotenv import load_dotenv
from tqdm.auto import tqdm
from openai import OpenAI

# tenacity: reintentos automáticos con back-off exponencial
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
)

load_dotenv()

pd.options.display.max_colwidth = None

print("OpenRouter:", os.environ["OPENROUTER_API_KEY"][:6])

OpenRouter: sk-or-


## Configuración general

Ajusta aquí las rutas, la semilla, el modelo y los providers activos.

In [30]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("data/dedo")          # carpeta con imágenes del dataset dedo
                                       # estructura esperada:
                                       #   data/dedo/si/<imagen>.jpg
                                       #   data/dedo/no/<imagen>.jpg

# ── Reproducibilidad ────────────────────────────────────────────────────────
SEED = 42
TEST_SIZE = 0.2

# ── Modelo OpenRouter ────────────────────────────────────────────────────────
# Lista de modelos multimodales gratuitos disponibles en OpenRouter:
# https://openrouter.ai/models?modality=image%2Btext
DEFAULT_OPENROUTER_MODEL = "google/gemini-2.0-flash-exp:free"

# ── Tenacity: política de reintentos ────────────────────────────────────────
MAX_RETRIES = 5
WAIT_MIN    = 2   # segundos
WAIT_MAX    = 30  # segundos

# ── Encoder visual para embeddings ──────────────────────────────────────────
DINO_MODEL_NAME = "vit_small_patch14_dinov2.lvd142m"

# ── N imágenes para prompting (limita coste de API) ─────────────────────────
N_PROMPTING = 30

## Inicializar cliente OpenRouter

In [31]:
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

## Cargar el dataset `dedo`

Se asume que las imágenes están organizadas en subcarpetas por clase:

```
data/dedo/
    si/   ← postura correcta
    no/   ← postura incorrecta
```

Si vuestra estructura es distinta, ajustad la función `load_dedo_dataset`.

In [32]:
def load_dedo_dataset(data_dir: Path) -> pd.DataFrame:
    """Carga imágenes del dataset dedo en un DataFrame."""
    rows = []
    label_map = {"si": "si", "no": "no"}

    for label_name, label_str in label_map.items():
        folder = data_dir / label_name
        if not folder.exists():
            print(f"[AVISO] No se encuentra la carpeta: {folder}")
            continue
        for img_path in sorted(folder.iterdir()):
            if img_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}:
                rows.append(
                    {
                        "path": img_path,
                        "label": label_str,
                        "label_id": 0 if label_str == "si" else 1,
                    }
                )

    df = pd.DataFrame(rows)
    print(f"Total imágenes cargadas: {len(df)}")
    print(df["label"].value_counts().to_string())
    return df


df_full = load_dedo_dataset(DATA_DIR)
df_full.head()

[AVISO] No se encuentra la carpeta: data/dedo/si
[AVISO] No se encuentra la carpeta: data/dedo/no
Total imágenes cargadas: 0


KeyError: 'label'

## Separación train / test

Usamos la misma semilla que en los días anteriores para que los resultados sean comparables.

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df_full,
    test_size=TEST_SIZE,
    stratify=df_full["label"],
    random_state=SEED,
)

df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"Train: {len(df_train)} | Test: {len(df_test)}")
print(df_train["label"].value_counts())

## Visualizar ejemplos del dataset

In [ ]:
def show_samples(df, n=8, title="Ejemplos"):
    sample = df.sample(n=min(n, len(df)), random_state=SEED).reset_index(drop=True)
    cols = 4
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 3.5))
    axes = axes.ravel()

    for ax, (_, row) in zip(axes, sample.iterrows()):
        img = Image.open(row["path"]).convert("RGB").resize((224, 224))
        ax.imshow(img)
        ax.set_title(f"label={row['label']}", fontsize=9)
        ax.axis("off")

    for ax in axes[len(sample):]:
        ax.axis("off")

    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


show_samples(df_test, n=8, title="Muestra del conjunto de test")

## Definir las clases en lenguaje natural

Para un modelo multimodal, `si` y `no` no son suficientes. Traducimos las etiquetas a una descripción visual precisa que el modelo pueda entender.

In [ ]:
label_names = {
    0: "si",
    1: "no",
}

label_descriptions = pd.DataFrame(
    {
        "label": ["si", "no"],
        "description": [
            (
                "postura correcta del dedo: el dedo está visible, centrado en el encuadre, "
                "bien iluminado y completamente contenido dentro de la zona iluminada, "
                "sin otras manos ni elementos que interfieran"
            ),
            (
                "postura incorrecta del dedo: el dedo está cortado, descentrado, mal iluminado, "
                "parcialmente fuera de la zona iluminada, o aparecen otras manos o elementos confusos"
            ),
        ],
    }
)

valid_labels = set(label_descriptions["label"])

label_descriptions

## Utilidades: imagen → base64 y construcción del prompt

In [ ]:
def pil_to_data_url(image: Image.Image, image_format: str = "PNG") -> str:
    """Convierte una imagen PIL a data URL en base64."""
    buffer = io.BytesIO()
    image.save(buffer, format=image_format)
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/{image_format.lower()};base64,{encoded}"


def path_to_pil(path: Path, size: int = 224) -> Image.Image:
    """Carga una imagen desde disco, la redimensiona y la convierte a RGB."""
    return Image.open(path).convert("RGB").resize((size, size))


def extract_json_object(text: str) -> dict:
    """Extrae el primer objeto JSON válido de un texto libre."""
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    assert match is not None, f"No se encontró JSON en: {text!r}"
    return json.loads(match.group(0))


SYSTEM_PROMPT = (
    "Eres un asistente experto en clasificación de imágenes de posicionamiento de dedos. "
    "Devuelve solo JSON válido, sin texto adicional."
)


def create_classification_prompt(label_df: pd.DataFrame) -> str:
    """Construye el prompt de clasificación a partir del DataFrame de etiquetas."""
    label_block = "\n".join(
        f"- {row['label']}: {row['description']}"
        for _, row in label_df.iterrows()
    )
    return f"""Clasifica la imagen del dedo en una de estas categorías:

{label_block}

Instrucciones:
- Analiza únicamente la imagen.
- No inventes categorías fuera de las indicadas.
- Si dudas, elige la categoría más probable.
- Devuelve solo un objeto JSON con este formato exacto:
{{"label": "si"}}
""".strip()


print(create_classification_prompt(label_descriptions))

## Clasificador zero-shot con OpenRouter y Tenacity

`tenacity` gestiona los reintentos automáticos cuando la API devuelve error de rate-limit o un error transitorio.
La política es **back-off exponencial**: espera 2 s, luego 4, 8 … hasta 30 s, con un máximo de 5 intentos.

In [ ]:
@retry(
    retry=retry_if_exception_type(Exception),
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=1, min=WAIT_MIN, max=WAIT_MAX),
    reraise=True,
)
def classify_with_openrouter(
    image: Image.Image,
    client: OpenAI =openrouter_client,
    model: str = DEFAULT_OPENROUTER_MODEL,
    label_df: pd.DataFrame = label_descriptions,
    system_prompt: str = SYSTEM_PROMPT,
) -> str:
    """
    Clasifica una imagen PIL usando un modelo multimodal via OpenRouter.

    Reintentos automáticos gestionados por tenacity:
    - máximo MAX_RETRIES intentos
    - espera exponencial entre WAIT_MIN y WAIT_MAX segundos
    """
    response = client.chat.completions.create(
        model=model,
        temperature=0.0,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": create_classification_prompt(label_df)},
                    {
                        "type": "image_url",
                        "image_url": {"url": pil_to_data_url(image)},
                    },
                ],
            },
        ],
    )

    raw = response.choices[0].message.content.strip()
    parsed = extract_json_object(raw)
    predicted_label = parsed["label"].strip().lower()

    if predicted_label not in valid_labels:
        raise ValueError(f"Etiqueta inesperada: {predicted_label!r}. Respuesta: {raw!r}")

    return predicted_label

## Probar el prompt con un ejemplo

In [ ]:
sample_row = df_test.iloc[0]
sample_image = path_to_pil(sample_row["path"])
sample_label = sample_row["label"]

plt.figure(figsize=(4, 4))
plt.imshow(sample_image)
plt.title(f"Etiqueta real: {sample_label}")
plt.axis("off")
plt.show()

prediction = classify_with_openrouter(sample_image)
print(f"Etiqueta real : {sample_label}")
print(f"Predicción    : {prediction}")
print(f"Correcto      : {sample_label == prediction}")

## Evaluación zero-shot en test

Evaluamos sobre un subconjunto de test para controlar el coste de API.
Ajusta `N_PROMPTING` en la celda de configuración según vuestro presupuesto.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


def evaluate_predictions(y_true, y_pred, labels):
    return {
        "accuracy":         accuracy_score(y_true, y_pred),
        "precision_macro":  precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro":     recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro":         f1_score(y_true, y_pred, average="macro", zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels),
    }


def classify_dataset(df, classifier_func, image_column="path", size=224):
    """Clasifica todas las imágenes de un DataFrame y devuelve las predicciones."""
    predictions = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Clasificando imágenes"):
        img = path_to_pil(row[image_column], size=size)
        predictions.append(classifier_func(img))

    return np.array(predictions)

In [ ]:
# Subconjunto de test para prompting
prompting_df = df_test.sample(n=min(N_PROMPTING, len(df_test)), random_state=SEED).reset_index(drop=True)

print(f"Imágenes a clasificar: {len(prompting_df)}")
print(prompting_df["label"].value_counts())

In [ ]:
y_pred_openrouter = classify_dataset(prompting_df, classify_with_openrouter)

metrics_openrouter = evaluate_predictions(
    prompting_df["label"].to_numpy(),
    y_pred_openrouter,
    labels=["si", "no"],
)

summary_df = pd.DataFrame([
    {
        "method":           "OpenRouter zero-shot",
        "model":            DEFAULT_OPENROUTER_MODEL,
        "accuracy":         round(metrics_openrouter["accuracy"], 3),
        "precision_macro":  round(metrics_openrouter["precision_macro"], 3),
        "recall_macro":     round(metrics_openrouter["recall_macro"], 3),
        "f1_macro":         round(metrics_openrouter["f1_macro"], 3),
    }
])

summary_df

## Matriz de confusión

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(
    metrics_openrouter["confusion_matrix"],
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["si", "no"],
    yticklabels=["si", "no"],
)
plt.xlabel("Predicción")
plt.ylabel("Etiqueta real")
plt.title(f"Matriz de confusión: OpenRouter zero-shot")
plt.tight_layout()
plt.show()

## Analizar respuestas y errores del prompting

Mirad los ejemplos donde el modelo falla para identificar:

- errores por prompt ambiguo;
- errores por imagen difícil o mal iluminada;
- errores por escena confusa (varias manos, fondo distractor).

In [ ]:
analysis_df = prompting_df[["path", "label"]].copy()
analysis_df["prediction"] = y_pred_openrouter
analysis_df["is_correct"] = analysis_df["label"] == analysis_df["prediction"]

print(f"Correctos : {analysis_df['is_correct'].sum()}")
print(f"Errores   : {(~analysis_df['is_correct']).sum()}")

mistakes_df = analysis_df[~analysis_df["is_correct"]].reset_index(drop=True)
mistakes_df[["path", "label", "prediction"]]

In [ ]:
n_show = min(4, len(mistakes_df))

if n_show == 0:
    print("¡Sin errores en este subconjunto!")
else:
    fig, axes = plt.subplots(1, n_show, figsize=(n_show * 3.5, 4))
    if n_show == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, mistakes_df.iloc[:n_show].iterrows()):
        img = path_to_pil(row["path"])
        ax.imshow(img)
        ax.set_title(f"real={row['label']}\npred={row['prediction']}", fontsize=9)
        ax.axis("off")

    plt.suptitle("Errores del modelo zero-shot", fontsize=12)
    plt.tight_layout()
    plt.show()

## Extensión opcional: few-shot

Añadir uno o dos ejemplos de referencia al prompt suele mejorar la precisión.
La función `classify_with_openrouter_fewshot` incluye una imagen correcta y una incorrecta antes de la imagen a clasificar.

In [ ]:
def build_few_shot_examples(df: pd.DataFrame, seed: int = SEED) -> dict:
    """
    Selecciona un ejemplo por clase para usar como referencia few-shot.
    Devuelve un dict {label: PIL.Image}.
    """
    examples = {}
    for label in ["si", "no"]:
        subset = df[df["label"] == label]
        if len(subset) == 0:
            continue
        row = subset.sample(1, random_state=seed).iloc[0]
        examples[label] = path_to_pil(row["path"])
    return examples


@retry(
    retry=retry_if_exception_type(Exception),
    stop=stop_after_attempt(MAX_RETRIES),
    wait=wait_exponential(multiplier=1, min=WAIT_MIN, max=WAIT_MAX),
    reraise=True,
)
def classify_with_openrouter_fewshot(
    image: Image.Image,
    few_shot_examples: dict,
    client: OpenAI = openrouter_client,
    model: str = DEFAULT_OPENROUTER_MODEL,
    label_df: pd.DataFrame = label_descriptions,
    system_prompt: str = SYSTEM_PROMPT,
) -> str:
    """Clasificación few-shot: incluye imágenes de ejemplo antes de la consulta."""
    # Construir los bloques de los ejemplos de referencia
    example_content = []
    for label_name, ref_image in few_shot_examples.items():
        example_content.append({"type": "text", "text": f"Ejemplo de postura '{label_name}':"})
        example_content.append({
            "type": "image_url",
            "image_url": {"url": pil_to_data_url(ref_image)},
        })

    # Añadir la imagen a clasificar
    example_content.append({"type": "text", "text": create_classification_prompt(label_df)})
    example_content.append({
        "type": "image_url",
        "image_url": {"url": pil_to_data_url(image)},
    })

    response = client.chat.completions.create(
        model=model,
        temperature=0.0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": example_content},
        ],
    )

    raw = response.choices[0].message.content.strip()
    parsed = extract_json_object(raw)
    predicted_label = parsed["label"].strip().lower()

    if predicted_label not in valid_labels:
        raise ValueError(f"Etiqueta inesperada: {predicted_label!r}. Respuesta: {raw!r}")

    return predicted_label

In [ ]:
# Seleccionar ejemplos de referencia del conjunto de entrenamiento
few_shot_examples = build_few_shot_examples(df_train)

# Mostrar los ejemplos que se usarán como referencia
fig, axes = plt.subplots(1, len(few_shot_examples), figsize=(8, 4))
for ax, (label_name, ref_img) in zip(axes, few_shot_examples.items()):
    ax.imshow(ref_img)
    ax.set_title(f"Referencia: {label_name}")
    ax.axis("off")
plt.suptitle("Imágenes de referencia few-shot")
plt.tight_layout()
plt.show()

# Evaluar few-shot sobre el mismo subconjunto
classify_fewshot = lambda img: classify_with_openrouter_fewshot(img, few_shot_examples)
y_pred_fewshot = classify_dataset(prompting_df, classify_fewshot)

metrics_fewshot = evaluate_predictions(
    prompting_df["label"].to_numpy(),
    y_pred_fewshot,
    labels=["si", "no"],
)

comparison_df = pd.DataFrame([
    {
        "method":           "Zero-shot",
        "accuracy":         round(metrics_openrouter["accuracy"], 3),
        "precision_macro":  round(metrics_openrouter["precision_macro"], 3),
        "recall_macro":     round(metrics_openrouter["recall_macro"], 3),
        "f1_macro":         round(metrics_openrouter["f1_macro"], 3),
    },
    {
        "method":           "Few-shot (1 ejemplo/clase)",
        "accuracy":         round(metrics_fewshot["accuracy"], 3),
        "precision_macro":  round(metrics_fewshot["precision_macro"], 3),
        "recall_macro":     round(metrics_fewshot["recall_macro"], 3),
        "f1_macro":         round(metrics_fewshot["f1_macro"], 3),
    },
])

comparison_df

## Extensión opcional: embeddings visuales con DINOv2

La segunda vía consiste en reutilizar un encoder visual preentrenado (DINOv2) como extractor de características y entrenar solo un clasificador lineal encima.

In [ ]:
import torch
import timm
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

dino_model = timm.create_model(
    DINO_MODEL_NAME,
    pretrained=True,
    num_classes=0,   # sin cabeza de clasificación → solo embeddings
)
dino_model = dino_model.to(device)
dino_model.eval()

dino_image_size = dino_model.patch_embed.img_size[0]
print(f"Tamaño de entrada DINOv2: {dino_image_size}x{dino_image_size}")

embedding_transform = transforms.Compose([
    transforms.Resize((dino_image_size, dino_image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

In [ ]:
def extract_embeddings(df: pd.DataFrame, model, transform, batch_size: int = 32) -> np.ndarray:
    """Extrae embeddings DINOv2 para todas las imágenes de un DataFrame."""
    all_embeddings = []

    with torch.inference_mode():
        for start in tqdm(range(0, len(df), batch_size), desc="Extrayendo embeddings"):
            batch_paths = df.iloc[start:start + batch_size]["path"].tolist()
            batch_images = [path_to_pil(p, size=dino_image_size) for p in batch_paths]
            batch_tensor = torch.stack([transform(img) for img in batch_images]).to(device)
            batch_embeddings = model(batch_tensor).cpu().numpy()
            all_embeddings.append(batch_embeddings)

    return np.vstack(all_embeddings)


X_train = extract_embeddings(df_train, dino_model, embedding_transform)
X_test  = extract_embeddings(df_test,  dino_model, embedding_transform)

y_train = df_train["label"].to_numpy()
y_test  = df_test["label"].to_numpy()

print(f"Train embeddings: {X_train.shape}")
print(f"Test  embeddings: {X_test.shape}")

In [ ]:
linear_clf = LogisticRegression(max_iter=2000, random_state=SEED)
linear_clf.fit(X_train, y_train)

emb_predictions = linear_clf.predict(X_test)
emb_metrics = evaluate_predictions(y_test, emb_predictions, labels=["si", "no"])

emb_summary = pd.DataFrame([{
    "method":           "DINOv2 + LogisticRegression",
    "accuracy":         round(emb_metrics["accuracy"], 3),
    "precision_macro":  round(emb_metrics["precision_macro"], 3),
    "recall_macro":     round(emb_metrics["recall_macro"], 3),
    "f1_macro":         round(emb_metrics["f1_macro"], 3),
}])

emb_summary

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(
    emb_metrics["confusion_matrix"],
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=["si", "no"],
    yticklabels=["si", "no"],
)
plt.xlabel("Predicción")
plt.ylabel("Etiqueta real")
plt.title("Matriz de confusión: DINOv2 + LogisticRegression")
plt.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
embedding_2d = pca.fit_transform(X_test)

plot_df = pd.DataFrame({
    "pc1":   embedding_2d[:, 0],
    "pc2":   embedding_2d[:, 1],
    "label": y_test,
})

plt.figure(figsize=(7, 5))
sns.scatterplot(data=plot_df, x="pc1", y="pc2", hue="label", alpha=0.8)
plt.title("Proyección PCA de embeddings DINOv2")
plt.tight_layout()
plt.show()

## Comparar enfoques

Si habéis completado más de una variante, esta celda resume las métricas de todos los métodos.

In [ ]:
all_results = []

# Zero-shot
all_results.append({
    "método":           "Zero-shot (OpenRouter)",
    "accuracy":         round(metrics_openrouter["accuracy"], 3),
    "precision_macro":  round(metrics_openrouter["precision_macro"], 3),
    "recall_macro":     round(metrics_openrouter["recall_macro"], 3),
    "f1_macro":         round(metrics_openrouter["f1_macro"], 3),
})

# Few-shot (si se ejecutó)
try:
    all_results.append({
        "método":           "Few-shot (OpenRouter)",
        "accuracy":         round(metrics_fewshot["accuracy"], 3),
        "precision_macro":  round(metrics_fewshot["precision_macro"], 3),
        "recall_macro":     round(metrics_fewshot["recall_macro"], 3),
        "f1_macro":         round(metrics_fewshot["f1_macro"], 3),
    })
except NameError:
    pass

# Embeddings (si se ejecutó)
try:
    all_results.append({
        "método":           "DINOv2 + LogisticRegression",
        "accuracy":         round(emb_metrics["accuracy"], 3),
        "precision_macro":  round(emb_metrics["precision_macro"], 3),
        "recall_macro":     round(emb_metrics["recall_macro"], 3),
        "f1_macro":         round(emb_metrics["f1_macro"], 3),
    })
except NameError:
    pass

final_df = pd.DataFrame(all_results).sort_values("f1_macro", ascending=False)
final_df

In [ ]:
if len(final_df) > 1:
    ax = final_df.set_index("método")[["accuracy", "f1_macro"]].plot(
        kind="bar", figsize=(9, 4), rot=15
    )
    ax.set_ylim(0, 1)
    ax.set_ylabel("Puntuación")
    ax.set_title("Comparación de enfoques: dataset dedo")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

## Conclusiones

Escribid aquí vuestras conclusiones:

- ¿Qué enfoque habéis completado?
- ¿Qué ventajas tiene respecto a entrenar una CNN desde cero?
- ¿Qué límites o errores habéis encontrado?
- ¿Qué mejoraríais si tuvierais más tiempo?